# ADAPEL: Adaptive Doubly-Robust Pseudo-outcome Ensemble Learner

**CATE meta-learning** — 4 steps:
1. Cross-fit (StratifiedKFold) → 2. Adaptive pseudo-outcome (DR + X, alpha-driven) →
3. R-Learner weighting (T-e)² → 4. NNLS positive stacking (4 base learners, L2 reg)

File duy nhất: **src** | **data** | **train** trong 1 notebook. Chạy Colab được (load data từ URL).

---
## SRC — Implementation

> ADAPEL class (375 dòng). Base learners: `HistGBM`, `ExtraTrees`, `Ridge`, `DecisionTree`.

In [10]:
from __future__ import annotations
import warnings
from abc import ABC, abstractmethod
from typing import Optional, Tuple
import numpy as np
from scipy.optimize import nnls
from sklearn.base import BaseEstimator, clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeRegressor
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")


class _PositiveStacking:
    """NNLS stacking weights >= 0, sparse when base learners correlate."""
    def __init__(self, coefs: np.ndarray, intercept: float = 0.0):
        self.coef_ = np.asarray(coefs, dtype=float)
        self.intercept_ = float(intercept)
    def predict(self, X: np.ndarray) -> np.ndarray:
        return X @ self.coef_ + self.intercept_


class BaseMetaLearner(ABC):
    @abstractmethod
    def fit(self, X, T, Y): ...
    @abstractmethod
    def predict(self, X): ...
    def estimate_ate(self, X): return float(np.mean(self.predict(X)))
    def estimate_att(self, X, T):
        return float(np.mean(self.predict(X[np.asarray(T).ravel() == 1])))
    def estimate_atc(self, X, T):
        return float(np.mean(self.predict(X[np.asarray(T).ravel() == 0])))


class ADAPEL(BaseMetaLearner):
    def __init__(
        self,
        outcome_estimator:    Optional[BaseEstimator] = None,
        propensity_estimator: Optional[BaseEstimator] = None,
        base_estimators:      Optional[list]          = None,
        n_folds:              int   = 3,
        fusion_gamma:         float = 1.0,
        min_alpha:            float = 0.1,
        clip_propensity:      float = 0.05,
    ) -> None:
        self.outcome_estimator = (
            outcome_estimator
            or HistGradientBoostingRegressor(random_state=42, max_iter=150, max_depth=6, learning_rate=0.05)
        )
        self.propensity_estimator = (
            propensity_estimator
            or HistGradientBoostingClassifier(random_state=42, max_iter=150, max_depth=6)
        )
        self.base_estimators = base_estimators or [
            HistGradientBoostingRegressor(random_state=42, max_iter=200, max_depth=5, learning_rate=0.05),
            ExtraTreesRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.7, n_jobs=-1, random_state=42),
            Ridge(alpha=1.0),
            DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42),
        ]
        self.n_folds         = n_folds
        self.fusion_gamma    = fusion_gamma
        self.min_alpha       = min_alpha
        self.clip_propensity = clip_propensity
        self._meta = self._fitted_finals = self._prop_full = None
        self._m0_full = self._m1_full = None
        self._bootstrap_learners = None
        self._t_res_std = 0.0

    def _alpha(self, e):
        raw = np.clip(1.0 - 4.0 * e * (1.0 - e), 0.0, 1.0) ** self.fusion_gamma
        return np.maximum(self.min_alpha, raw)
    def _clip_e(self, e):
        return np.clip(e, self.clip_propensity, 1.0 - self.clip_propensity)
    @staticmethod
    def _fit_w(est, X, y, sw):
        try:    return est.fit(X, y, sample_weight=sw)
        except TypeError: return est.fit(X, y)

    def _fit_positive_stacking(self, oof, pseudo, sw) -> _PositiveStacking:
        nb = oof.shape[1]
        sw_n = sw / max(sw.mean(), 1e-10)
        sqrt_w = np.sqrt(np.maximum(sw_n, 1e-10))
        A = np.column_stack([oof * sqrt_w[:, None], sqrt_w])
        b = pseudo * sqrt_w
        lam = 1e-3 * b.std() / nb
        A_reg = np.column_stack([np.eye(nb) * np.sqrt(lam), np.zeros(nb)])
        A = np.vstack([A, A_reg])
        b = np.concatenate([b, np.zeros(nb)])
        coefs, _ = nnls(A, b, maxiter=500 * nb)
        base_w, intercept = coefs[:nb], coefs[nb]
        if base_w.sum() < 1e-6:
            base_w = np.ones(nb) / nb
            intercept = 0.0
        return _PositiveStacking(base_w, intercept)

    def _validate(self, X, T, Y):
        X = np.atleast_2d(np.asarray(X, dtype=float))
        T = np.asarray(T, dtype=float).ravel()
        Y = np.asarray(Y, dtype=float).ravel()
        assert X.shape[0] == T.shape[0] == Y.shape[0], "X, T, Y rows differ"
        assert set(np.unique(T)).issubset({0.0, 1.0}), "T must be binary 0/1"
        return X, T, Y
    def _check_fitted(self):
        if self._fitted_finals is None:
            raise RuntimeError("ADAPEL not fitted. Call .fit() first.")

    def fit(self, X, T, Y) -> "ADAPEL":
        X, T, Y = self._validate(X, T, Y)
        n, nb = X.shape[0], len(self.base_estimators)
        pDR, pX, ehat, tres = np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
        n_folds = max(3, min(self.n_folds, int(n / 100)))
        skf = StratifiedKFold(n_folds, shuffle=True, random_state=42)
        for tr, val in skf.split(X, T):
            Xtr, Xv, Ttr, Tv, Ytr, Yv = X[tr], X[val], T[tr], T[val], Y[tr], Y[val]
            i0, i1 = Ttr == 0, Ttr == 1
            m0 = clone(self.outcome_estimator)
            m1 = clone(self.outcome_estimator)
            if i0.sum() >= 5 and i1.sum() >= 5:
                m0.fit(Xtr[i0], Ytr[i0]); m1.fit(Xtr[i1], Ytr[i1])
            else:
                m0.fit(Xtr, Ytr); m1.fit(Xtr, Ytr)
            mu0, mu1 = m0.predict(Xv), m1.predict(Xv)
            e = self._clip_e(clone(self.propensity_estimator).fit(Xtr, Ttr).predict_proba(Xv)[:, 1])
            ehat[val] = e; tres[val] = Tv - e
            pDR[val] = (mu1 - mu0) + (Tv - e) / (e * (1 - e)) * (Yv - np.where(Tv == 1, mu1, mu0))
            pX[val]  = np.where(Tv == 1, Yv - mu0, mu1 - Yv)
        a      = self._alpha(ehat)
        pseudo = a * pX + (1.0 - a) * pDR
        sw     = tres ** 2
        sw    /= max(sw.mean(), 1e-10)
        oof = np.zeros((n, nb))
        for j, est in enumerate(self.base_estimators):
            for tr2, val2 in skf.split(X, T):
                oof[val2, j] = self._fit_w(clone(est), X[tr2], pseudo[tr2], sw[tr2]).predict(X[val2])
        self._meta          = self._fit_positive_stacking(oof, pseudo, sw)
        self._fitted_finals = [self._fit_w(clone(m), X, pseudo, sw) for m in self.base_estimators]
        self._prop_full     = clone(self.propensity_estimator).fit(X, T)
        self._t_res_std     = float(tres.std())
        i0f, i1f = T == 0, T == 1
        self._m0_full = clone(self.outcome_estimator).fit(X[i0f], Y[i0f])
        self._m1_full = clone(self.outcome_estimator).fit(X[i1f], Y[i1f])
        return self

    def predict(self, X) -> np.ndarray:
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        return self._meta.predict(np.column_stack([m.predict(X) for m in self._fitted_finals]))
    def predict_potential_outcomes(self, X) -> Tuple[np.ndarray, np.ndarray]:
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        return self._m0_full.predict(X), self._m1_full.predict(X)
    def predict_counterfactual(self, X, T_observed) -> np.ndarray:
        X = np.atleast_2d(np.asarray(X, dtype=float))
        T_obs = np.asarray(T_observed).ravel()
        assert X.shape[0] == T_obs.shape[0]
        y0, y1 = self.predict_potential_outcomes(X)
        return np.where(T_obs == 1, y0, y1)

    def get_diagnostics(self, X) -> dict:
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        e = self._clip_e(self._prop_full.predict_proba(X)[:, 1])
        a = self._alpha(e)
        return {
            "propensity": e, "alpha": a,
            "meta_weights": self._meta.coef_,
            "pct_dr_dominant": float((a < 0.5).mean()),
            "pct_x_dominant": float((a >= 0.5).mean()),
            "ensemble_std": np.column_stack([m.predict(X) for m in self._fitted_finals]).std(axis=1),
            "t_res_std_train": self._t_res_std,
        }

    def fit_bootstrap(self, X, T, Y, n_bootstrap: int = 10, random_state: int = 42,
                      n_jobs: int = -1) -> "ADAPEL":
        X, T, Y = self._validate(X, T, Y)
        n = X.shape[0]
        self.fit(X, T, Y)
        light_outcome = self._lighten(self.outcome_estimator)
        light_prop    = self._lighten(self.propensity_estimator)
        light_finals  = [self._lighten(m) for m in self.base_estimators]
        def _fit_one(seed):
            rng = np.random.default_rng(seed)
            idx = rng.choice(n, size=n, replace=True)
            Xb, Tb, Yb = X[idx], T[idx], Y[idx]
            if Tb.sum() < 5 or (1 - Tb).sum() < 5:
                return None
            bl = ADAPEL(
                outcome_estimator=clone(light_outcome),
                propensity_estimator=clone(light_prop),
                base_estimators=[clone(m) for m in light_finals],
                n_folds=self.n_folds, fusion_gamma=self.fusion_gamma,
                min_alpha=self.min_alpha, clip_propensity=self.clip_propensity,
            )
            try:
                bl.fit(Xb, Tb, Yb); return bl
            except Exception:
                return None
        seeds = [random_state + i + 1 for i in range(n_bootstrap)]
        results = Parallel(n_jobs=n_jobs)(delayed(_fit_one)(s) for s in seeds)
        self._bootstrap_learners = [r for r in results if r is not None]
        return self

    @staticmethod
    def _lighten(est) -> BaseEstimator:
        el = clone(est)
        for attr in ("max_iter", "n_estimators"):
            if hasattr(el, attr) and getattr(el, attr) is not None:
                setattr(el, attr, min(getattr(el, attr), 80))
        if hasattr(el, "n_jobs"): el.n_jobs = -1
        return el

    def predict_clinical(self, X, alpha: float = 0.05) -> dict:
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        cate = self.predict(X)
        e_raw = self._prop_full.predict_proba(X)[:, 1]
        in_overlap = (e_raw >= self.clip_propensity) & (e_raw <= 1.0 - self.clip_propensity)
        e = self._clip_e(e_raw)
        lower = upper = std = None
        if self._bootstrap_learners:
            preds = np.column_stack([m.predict(X) for m in self._bootstrap_learners])
            cate = preds.mean(axis=1)
            lower = np.percentile(preds, 100 * alpha / 2, axis=1)
            upper = np.percentile(preds, 100 * (1 - alpha / 2), axis=1)
            std = preds.std(axis=1)
        return {"cate": cate, "lower_ci": lower, "upper_ci": upper,
                "bootstrap_std": std, "propensity": e,
                "propensity_raw": e_raw, "in_overlap": in_overlap}

    def estimate_e_value(self, X, outcome_type: str = "binary") -> float:
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        ate = self.estimate_ate(X)
        if outcome_type == "binary":
            p0 = float(np.clip(np.mean(self._m0_full.predict(X)), 1e-5, 1 - 1e-5))
            p1 = float(np.clip(np.mean(self._m1_full.predict(X)), 1e-5, 1 - 1e-5))
            rr = max(p1 / p0, p0 / p1)
        else:
            y0, y1 = self.predict_potential_outcomes(X)
            std = max(float(np.std(np.concatenate([y0, y1]))), 1e-5)
            rr = np.exp(0.91 * abs(ate / std))
        return 1.0 if rr <= 1.0 else float(rr + np.sqrt(rr * (rr - 1.0)))

    def explain_cate_surrogate(self, X, feature_names: Optional[list] = None, max_depth: int = 3) -> str:
        from sklearn.tree import export_text
        self._check_fitted()
        X = np.atleast_2d(np.asarray(X, dtype=float))
        surrogate = DecisionTreeRegressor(max_depth=max_depth, random_state=42).fit(X, self.predict(X))
        names = feature_names or [f"F{i}" for i in range(X.shape[1])]
        out = []
        for line in export_text(surrogate, feature_names=names).split("\n"):
            if line.strip(): out.append(line)
        return "\n".join(out)

---
## DATA — Dataset Loaders

> Tải từ URL (Colab-friendly). 7 datasets: IHDP, IHDP-100, RHC, Lalonde, Hillstrom, ACIC 2016, Twins.

In [11]:
import os, ssl, io, urllib, zipfile
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from pandas.api.types import is_numeric_dtype
from sklearn.base import clone

def _urlread(url):
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    return urllib.request.urlopen(url, context=ctx).read()
def _download_csv(url):
    return pd.read_csv(io.BytesIO(_urlread(url)))
def _download_npz(url):
    return np.load(io.BytesIO(_urlread(url)))

def benchmark_t_learner(X, T, Y, true_cate=None, n_splits=5, seed=42):
    pred = np.zeros(len(X))
    gbm = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=seed)
    for tr, val in KFold(n_splits, shuffle=True, random_state=seed).split(X):
        m = clone(gbm).fit(np.column_stack([X[tr], T[tr]]), Y[tr])
        pred[val] = m.predict(np.column_stack([X[val], np.ones(len(val))])) - m.predict(np.column_stack([X[val], np.zeros(len(val))]))
    return np.sqrt(np.mean((pred - true_cate) ** 2)) if true_cate is not None else pred.mean()

def load_ihdp_single():
    data = pd.read_csv(io.StringIO(_urlread(
        "https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/IHDP/csv/ihdp_npci_1.csv"
    ).decode())).values
    T, Y, mu0, mu1 = data[:, 0], data[:, 1], data[:, 3], data[:, 4]
    return data[:, 5:], T, Y, mu1 - mu0

def load_ihdp100():
    return (_download_npz("https://www.fredjo.com/files/ihdp_npci_1-100.train.npz"),
            _download_npz("https://www.fredjo.com/files/ihdp_npci_1-100.test.npz"))

def load_rhc():
    df = _download_csv("https://hbiostat.org/data/repo/rhc.csv")
    T = (df["swang1"] == "RHC").astype(int).values
    Y = (df["death"] == "Yes").astype(int).values
    cov_cols = ["age","sex","race","edu","income","ninsclas","cat1","das2d3pc","dnr1","ca",
        "surv2md1","aps1","scoma1","meanbp1","wblc1","hrt1","resp1","temp1","pafi1","alb1",
        "hema1","bili1","crea1","sod1","pot1","paco21","ph1","cardiohx","chfhx","dementhx",
        "psychhx","chrpulhx","renalhx","liverhx","gibledhx","malighx","immunhx","transhx","amihx"]
    X_df = df[cov_cols].copy()
    for col in X_df.columns:
        if is_numeric_dtype(X_df[col]): X_df[col] = X_df[col].fillna(X_df[col].median())
        else: X_df[col] = X_df[col].fillna(X_df[col].mode()[0])
    return pd.get_dummies(X_df, drop_first=True).astype(float).values, T, Y

def load_lalonde():
    df = _download_csv("https://raw.githubusercontent.com/robjellis/lalonde/master/lalonde_data.csv")
    X = df[["age","educ","black","hispan","married","nodegree","re74","re75"]].values
    return X, df["treat"].values, df["re78"].values / 1000.0

def load_hillstrom():
    df = _download_csv("https://raw.githubusercontent.com/W-Tran/uplift-modelling/master/data/hillstrom/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv")
    T = np.where(df["segment"] != "No E-Mail", 1, 0)
    X_df = df[["recency","history","mens","womens","newbie"]].copy()
    for name in ("zip_code", "channel"):
        X_df = pd.concat([X_df, pd.get_dummies(df[name], prefix=name, drop_first=True)], axis=1)
    return X_df.astype(float).values, T, df["spend"].values

def load_acic2016(rep):
    X_df = pd.get_dummies(
        _download_csv("https://raw.githubusercontent.com/BiomedSciAI/causallib/master/causallib/datasets/data/acic_challenge_2016/x.csv"),
        drop_first=True)
    z = _download_csv(f"https://raw.githubusercontent.com/BiomedSciAI/causallib/master/causallib/datasets/data/acic_challenge_2016/zymu_{rep}.csv")
    T, Y = z["z"].values, np.where(z["z"].values == 1, z["y1"].values, z["y0"].values)
    return X_df.values.astype(float), T, Y, z["mu1"].values - z["mu0"].values

print("All loaders ready. OK")

All loaders ready. OK


---
## TRAIN — IHDP (single realization)

| Metric | Value |
|---|---|
| **Reference** | Hill (2011) — JCGS |
| **Type** | Semi-synthetic (real covariates, synthetic outcome) |
| **Samples** | 747 (1 realization of NPCI) |
| **Features** | 25 (6 confounders + 19 noise) |
| **Treatment rate** | ~19% |
| **Outcome** | Continuous (cognitive test score) |
| **Ground truth** | CATE = mu1(x) - mu0(x) (known) |
| **Task** | PEHE (Precision in Estimation of Heterogeneous Effect) |

In [12]:
print("=" * 60)
print("IHDP (single)")
print("=" * 60)
X, T, Y, true_cate = load_ihdp_single()
print(f"Shape: {X.shape}  T-rate: {T.mean():.1%}  True ATE: {true_cate.mean():.4f}")
pehe_t = benchmark_t_learner(X, T, Y, true_cate)
model = ADAPEL(n_folds=5, fusion_gamma=2.0, min_alpha=0.0).fit(X, T, Y)
pehe_a = np.sqrt(np.mean((model.predict(X) - true_cate) ** 2))
d = model.get_diagnostics(X)
print(f"T-Learner PEHE: {pehe_t:.4f}")
print(f"ADAPEL    PEHE: {pehe_a:.4f}")
print(f"Stacking weights: {d['meta_weights']}  sum={d['meta_weights'].sum():.3f}")
print(f"DR dominant: {d['pct_dr_dominant']:.1%}  X dominant: {d['pct_x_dominant']:.1%}")

IHDP (single)
Shape: (746, 25)  T-rate: 18.5%  True ATE: 4.0166
T-Learner PEHE: 0.6407
ADAPEL    PEHE: 2.2710
Stacking weights: [0.25 0.25 0.25 0.25]  sum=1.000
DR dominant: 38.6%  X dominant: 61.4%


---
## TRAIN — IHDP-100 (100 realizations)

| Metric | Value |
|---|---|
| **Reference** | NPCI benchmark (fredjo.com) |
| **Type** | Semi-synthetic (100 different response surfaces) |
| **Samples** | 672 train + 75 test × 100 reps |
| **Features** | 25 |
| **Treatment rate** | ~19% per rep |
| **Outcome** | Continuous |
| **Ground truth** | CATE known (mu1 - mu0 from simulation) |
| **Task** | Mean PEHE across 100 reps |
| **⏱** | ~15-30 phút nếu chạy đủ 100 reps |

> 👇 Mặc định `n_reps=10` cho nhanh. Set `n_reps=100` để full benchmark.

In [13]:
print("=" * 60)
print("IHDP-100")
print("=" * 60)
train, test = load_ihdp100()
n_reps = 10  # dat 100 cho full benchmark
print(f"Running {n_reps} reps...")
gbm = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
scores = {"T-Learner": [], "S-Learner": [], "ADAPEL": []}
for rep in range(n_reps):
    X_tr, T_tr, Y_tr = train["x"][:, :, rep], train["t"][:, rep], train["yf"][:, rep]
    X_te, true_te = test["x"][:, :, rep], test["mu1"][:, rep] - test["mu0"][:, rep]
    m = clone(gbm).fit(np.column_stack([X_tr, T_tr]), Y_tr)
    p = m.predict(np.column_stack([X_te, np.ones(len(X_te))])) - m.predict(np.column_stack([X_te, np.zeros(len(X_te))]))
    scores["T-Learner"].append(np.sqrt(np.mean((p - true_te) ** 2)))
    m0 = clone(gbm).fit(X_tr[T_tr == 0], Y_tr[T_tr == 0])
    scores["S-Learner"].append(np.sqrt(np.mean((-m0.predict(X_te) - true_te) ** 2)))
    m_ad = ADAPEL(n_folds=3, fusion_gamma=2.0, min_alpha=0.0).fit(X_tr, T_tr, Y_tr)
    scores["ADAPEL"].append(np.sqrt(np.mean((m_ad.predict(X_te) - true_te) ** 2)))
    if (rep + 1) % 5 == 0:
        print(f"  Rep {rep+1}/{n_reps}")
print()
for name, vals in scores.items():
    arr = np.array(vals)
    print(f"{name:12s} mean={arr.mean():.4f} std={arr.std():.4f}")

IHDP-100
Running 10 reps...
  Rep 5/10
  Rep 10/10

T-Learner    mean=3.0342 std=6.0248
S-Learner    mean=14.2139 std=13.5130
ADAPEL       mean=3.9125 std=6.0765


---
## TRAIN — RHC (Real Observational)

| Metric | Value |
|---|---|
| **Reference** | Connors et al. (1996) — JAMA |
| **Type** | Real observational (critical care) |
| **Samples** | 5,735 |
| **Features** | 39 → 54 after one-hot |
| **Treatment** | RHC (Right Heart Catheterization) — 38.1% |
| **Outcome** | Binary — 30-day mortality (24.9%) |
| **Ground truth** | Unknown (no RCT on RHC) |
| **Task** | ATE estimation + E-Value sensitivity |
| **Expected** | RHC increases mortality (Connors ATE ~ +5%) |

In [14]:
print("=" * 60)
print("RHC Clinical")
print("=" * 60)
X, T, Y = load_rhc()
print(f"{X.shape[0]} patients, {X.shape[1]} features, RHC rate: {T.mean():.1%}")
print(f"Naive RD: {Y[T==1].mean():.3f} - {Y[T==0].mean():.3f} = {Y[T==1].mean() - Y[T==0].mean():.4f} (confounded)")
model = ADAPEL(n_folds=3, fusion_gamma=1.0, min_alpha=0.1, clip_propensity=0.05)
model.fit_bootstrap(X, T, Y, n_bootstrap=15, random_state=42)
clin = model.predict_clinical(X)
ate, e_val = clin["cate"].mean(), model.estimate_e_value(X, 'binary')
d = model.get_diagnostics(X)
print(f"ADAPEL ATE: {ate:.4f}  E-Value: {e_val:.4f}")
print(f"Weights: {d['meta_weights']}")
print(f"Interpretation: RHC {'INCREASES' if ate>0 else 'DECREASES'} mortality by {abs(ate)*100:.2f}%")

RHC Clinical
5735 patients, 54 features, RHC rate: 38.1%
Naive RD: 0.680 - 0.630 = 0.0507 (confounded)
ADAPEL ATE: 0.0517  E-Value: 1.3320
Weights: [0.25 0.25 0.25 0.25]
Interpretation: RHC INCREASES mortality by 5.17%


---
## TRAIN — Lalonde (RCT Benchmark)

| Metric | Value |
|---|---|
| **Reference** | LaLonde (1986) — AER |
| **Type** | RCT (NSW labor training) |
| **Samples** | 614 (NSW treated + PSID controls) |
| **Features** | 8 (age, educ, race, married, earnings...) |
| **Treatment** | Job training — ~22% |
| **Outcome** | Continuous — real earnings 1978 ($1000s) |
| **Ground truth** | RCT ATE ~ $0.9k (varies by sample) |
| **Task** | ATE gần bằng RCT benchmark |

In [15]:
print("=" * 60)
print("Lalonde")
print("=" * 60)
X, T, Y = load_lalonde()
ate_rct = Y[T==1].mean() - Y[T==0].mean()
print(f"{X.shape[0]} samples, {X.shape[1]} features, T-rate: {T.mean():.1%}")
print(f"RCT ATE (diff-in-means): {ate_rct:.3f}k")
model = ADAPEL(n_folds=3).fit(X, T, Y)
print(f"ADAPEL ATE: {model.estimate_ate(X):.4f}k  diff: {abs(model.estimate_ate(X) - ate_rct):.4f}k")
print(f"Weights: {model.get_diagnostics(X)['meta_weights']}")

Lalonde
614 samples, 8 features, T-rate: 30.1%
RCT ATE (diff-in-means): -0.635k
ADAPEL ATE: 0.1260k  diff: 0.7610k
Weights: [0.01001671 0.         0.         0.        ]


---
## TRAIN — Hillstrom (RCT Benchmark)

| Metric | Value |
|---|---|
| **Reference** | MineThatData e-mail analytics challenge |
| **Type** | RCT (A/B test) |
| **Samples** | 64,000 |
| **Features** | 10 (recency, history, zip, channel, ...) |
| **Treatment** | Email (Mens or Womens) — 66.7% |
| **Outcome** | Continuous — spend ($) |
| **Ground truth** | RCT ATE = unbiased diff-in-means |
| **Task** | ADAPEL ATE ≈ RCT ATE (diff < 0.01) |

In [16]:
print("=" * 60)
print("Hillstrom (RCT)")
print("=" * 60)
X, T, Y = load_hillstrom()
ate_rct = Y[T==1].mean() - Y[T==0].mean()
print(f"{X.shape[0]} samples, T-rate: {T.mean():.1%}")
print(f"RCT ATE: {ate_rct:.4f}")
model = ADAPEL(n_folds=3).fit(X, T, Y)
ate_ad = model.estimate_ate(X)
print(f"ADAPEL ATE: {ate_ad:.4f}  diff from RCT: {abs(ate_ad - ate_rct):.4f}")

Hillstrom (RCT)
64000 samples, T-rate: 66.7%
RCT ATE: 0.5968
ADAPEL ATE: 0.6033  diff from RCT: 0.0065


---
## TRAIN — ACIC 2016 (10 settings)

| Metric | Value |
|---|---|
| **Reference** | Causal Inference Challenge (2016) |
| **Type** | Semi-synthetic (real X, synthetic Y) |
| **Samples** | 4,802 (80/20 train/test) |
| **Features** | 58 after one-hot encoding |
| **Treatment** | Varies per setting (~30-50%) |
| **Outcome** | Continuous (10 different zymu responses) |
| **Ground truth** | CATE known (mu1 - mu0) |
| **Task** | Mean PEHE across 10 settings |

In [ ]:
print("=" * 60)
print("ACIC 2016")
print("=" * 60)
gbm = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
scores = {"T-Learner": [], "ADAPEL": []}
for setting in range(1, 11):
    X_all, T, Y, true_cate = load_acic2016(setting)
    rng = np.random.default_rng(setting)
    idx = rng.permutation(len(X_all))
    tr, te = idx[:int(len(X_all)*0.8)], idx[int(len(X_all)*0.8):]
    scaler = StandardScaler()
    X_tr, X_te = scaler.fit_transform(X_all[tr]), scaler.transform(X_all[te])
    m = clone(gbm).fit(np.column_stack([X_tr, T[tr]]), Y[tr])
    p = m.predict(np.column_stack([X_te, np.ones(len(X_te))])) - m.predict(np.column_stack([X_te, np.zeros(len(X_te))]))
    scores["T-Learner"].append(np.sqrt(np.mean((p - true_cate[te]) ** 2)))
    m_ad = ADAPEL(n_folds=3, fusion_gamma=1.0, min_alpha=0.1, clip_propensity=0.05).fit(X_tr, T[tr], Y[tr])
    scores["ADAPEL"].append(np.sqrt(np.mean((m_ad.predict(X_te) - true_cate[te]) ** 2)))
    print(f"  Setting {setting:2d}: T-Learner={scores['T-Learner'][-1]:.3f}  ADAPEL={scores['ADAPEL'][-1]:.3f}")
print()
for name, vals in scores.items():
    arr = np.array(vals)
    print(f"{name:12s} mean={arr.mean():.4f} std={arr.std():.4f}")

ACIC 2016
  Setting  1: T-Learner=1.024  ADAPEL=1.134
  Setting  2: T-Learner=0.697  ADAPEL=0.851
  Setting  3: T-Learner=0.785  ADAPEL=0.642


---
## TRAIN — Synthetic Quick Test

> Chạy nhanh (không cần download data). Test end-to-end: fit → predict → bootstrap → CI → E-Value.

In [ ]:
print("=" * 60)
print("Synthetic Quick Test")
print("=" * 60)
rng = np.random.default_rng(42)
X = rng.normal(0, 1, (200, 5))
T = rng.binomial(1, 0.3, 200)
Y = 0.5 * X[:, 0] + 0.3 * T * X[:, 1] + rng.normal(0, 0.5, 200)
model = ADAPEL(n_folds=3).fit(X, T, Y)
print(f"ATE: {model.estimate_ate(X):.4f}")
print(f"Weights: {model.get_diagnostics(X)['meta_weights']}")
model.fit_bootstrap(X, T, Y, n_bootstrap=5)
clin = model.predict_clinical(X)
print(f"Bootstrap CI coverage (all True = OK): {(clin['lower_ci'] <= clin['cate']).all()}")
print(f"E-Value: {model.estimate_e_value(X, 'continuous'):.4f}")
print("All OK.")